# Gateway vs Azure Judge — Single GT Latency Check

Runs one existing ground-truth case through both official judge backends and compares application-visible evaluation latency and coverage decisions.

GT source: row `0` from `golden_set_augmented_tagged.csv`.

Fields used:
- context: `theme_businessNeeds`, `theme_description`
- output: `epic_description`, `epic_successCriteria`

This notebook performs one `CoverageEvaluator` call per backend. It is a smoke comparison, not a statistically meaningful performance benchmark.


In [ ]:
from pathlib import Path
import time

import pandas as pd

from idp_eval import (
    CoverageEvaluator,
    EvaluationCase,
    EvaluationFramework,
    create_azure_judge,
    create_gateway_judge,
)
from idp_eval.judges import AzureJudgeConfig, GatewayJudgeConfig


## 1. Load the exact GT row

Provide `golden_set_augmented_tagged.csv` locally or update `GOLDEN_SET_PATH`. The dataset is not stored in this repository.


In [ ]:
GOLDEN_SET_PATH = Path("golden_set_augmented_tagged.csv")
GT_ROW_INDEX = 0

df = pd.read_csv(GOLDEN_SET_PATH).fillna("")
row = df.iloc[GT_ROW_INDEX]

case = EvaluationCase(
    case_id="backend-latency-gt-row-0",
    context={
        "theme_business_needs": row.get("theme_businessNeeds", ""),
        "theme_description": row.get("theme_description", ""),
    },
    output={
        "epic_description": row.get("epic_description", ""),
        "epic_success_criteria": row.get("epic_successCriteria", ""),
    },
)

case


## 2. Create both judges

For imported production use, passing an application-owned config object is the recommended path. Environment variables and optional YAML remain supported conveniences, but this notebook uses explicit config injection to mirror how another application typically consumes the package.

Replace the placeholders locally, or populate these objects from your application's existing settings and secrets layer. Never commit real configuration values to this notebook. A setup failure for one backend does not prevent testing the other.


In [ ]:
gateway_config = GatewayJudgeConfig(
    model="...",
    base_url="...",
    app_id="...",
    idp_auth_url="...",
    idp_client_id="...",
    idp_client_secret="...",
    idp_user="...",
    idp_password="...",
    verify_ssl=True,
    timeout=90,
)

azure_config = AzureJudgeConfig(
    model="...",
    azure_endpoint="...",
    tenant_id="...",
    client_id="...",
    client_secret="...",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)

gateway_judge = None
azure_judge = None
setup_errors = {}

try:
    gateway_judge = create_gateway_judge(config=gateway_config)
except Exception as exc:
    setup_errors["gateway"] = f"{type(exc).__name__}: {exc}"

try:
    azure_judge = create_azure_judge(config=azure_config)
except Exception as exc:
    setup_errors["azure"] = f"{type(exc).__name__}: {exc}"

print("Gateway model:", getattr(gateway_judge, "model", None))
print("Azure model:  ", getattr(azure_judge, "model", None))
for backend, error in setup_errors.items():
    print(f"{backend} setup failed: {error}")


## 3. Run the same CoverageEvaluator once on each backend

`time.perf_counter()` measures `framework.evaluate(case)` end to end. `verbose=True` retains the item-level audit trail.


In [ ]:
def run_once(name, judge, case, setup_error=None):
    if judge is None:
        return {
            "backend": name,
            "model": None,
            "latency_s": None,
            "score": None,
            "label": None,
            "item_count": None,
            "judge_call_count": None,
            "result": None,
            "error": setup_error,
        }

    evaluator = CoverageEvaluator(judge, verbose=True)
    framework = EvaluationFramework(judge=judge, evaluators=[evaluator])
    started = time.perf_counter()
    try:
        result = framework.evaluate(case)["coverage"]
        latency_s = time.perf_counter() - started
        return {
            "backend": name,
            "model": getattr(judge, "model", None),
            "latency_s": latency_s,
            "score": result.score,
            "label": result.label,
            "item_count": result.details.get("final_item_count"),
            "judge_call_count": result.details.get("judge_call_count"),
            "result": result,
            "error": None,
        }
    except Exception as exc:
        return {
            "backend": name,
            "model": getattr(judge, "model", None),
            "latency_s": time.perf_counter() - started,
            "score": None,
            "label": None,
            "item_count": None,
            "judge_call_count": None,
            "result": None,
            "error": f"{type(exc).__name__}: {exc}",
        }

runs = [
    run_once("gateway", gateway_judge, case, setup_errors.get("gateway")),
    run_once("azure", azure_judge, case, setup_errors.get("azure")),
]


In [ ]:
comparison = pd.DataFrame([
    {
        "backend": run["backend"],
        "model": run["model"],
        "latency_s": (
            round(run["latency_s"], 3)
            if run["latency_s"] is not None
            else None
        ),
        "score": run["score"],
        "label": run["label"],
        "item_count": run["item_count"],
        "judge_call_count": run["judge_call_count"],
        "error": run["error"],
    }
    for run in runs
])

comparison


## 4. Compare item-level decisions


In [ ]:
item_columns = [
    "source_item",
    "meaningfully_present",
    "fully_present",
    "status",
    "item_score",
    "reason",
]

for run in runs:
    print(f"\n=== {run['backend'].upper()} ===")
    if run["error"]:
        print(run["error"])
        continue

    result = run["result"]
    print(f"latency_s={run['latency_s']:.3f}")
    print(f"score={result.score} label={result.label}")
    items = pd.DataFrame(result.details.get("items", []))
    display(items.reindex(columns=item_columns))


## 5. Close judge resources


In [ ]:
if gateway_judge is not None:
    gateway_judge.close()
if azure_judge is not None:
    azure_judge.close()
print("Judge resources closed.")
